In [1]:
import pandas as pd
import numpy as np

ARCHIVO = "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA = "ventas_conjunto_detalle"
DIAS_ANALISIS_3M = 90

# =========================
# CARGA
# =========================
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# =========================
# VALIDACIONES
# =========================
campos_requeridos = ["fecha", "cantidad", "sku_madre", "canal"]
faltantes = [c for c in campos_requeridos if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")

# =========================
# LIMPIEZA
# =========================
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)

# Solo ventas vinculadas a SKU madre
if "tiene_referencia_madre" in df.columns:
    df = df[df["tiene_referencia_madre"].astype(str).str.upper().eq("SI")].copy()

df = df[
    df["sku_madre"].notna()
    & df["sku_madre"].astype(str).str.strip().ne("")
    & df["fecha"].notna()
].copy()

# =========================
# FILTRO ÚLTIMOS 3 MESES
# =========================
fecha_fin = df["fecha"].max().normalize()
fecha_inicio = fecha_fin - pd.Timedelta(days=DIAS_ANALISIS_3M - 1)

df_3m = df[
    (df["fecha"] >= fecha_inicio)
    & (df["fecha"] <= fecha_fin + pd.Timedelta(days=1))
].copy()

# =========================
# EXCLUIR FULL / DROP
# =========================
df_3m["canal"] = df_3m["canal"].astype(str).str.strip()
df_3m = df_3m[~df_3m["canal"].str.lower().isin(["full", "drop"])].copy()

# =========================
# AGRUPACIÓN POR SKU MADRE Y CANAL
# =========================
resumen = (
    df_3m.groupby(["sku_madre", "canal"], as_index=False)
    .agg(ventas_canal=("cantidad", "sum"))
)

# Total por SKU madre
resumen["ventas_totales_sku"] = resumen.groupby("sku_madre")["ventas_canal"].transform("sum")

# Porcentaje por canal
resumen["porcentaje_canal"] = np.where(
    resumen["ventas_totales_sku"] > 0,
    resumen["ventas_canal"] / resumen["ventas_totales_sku"] * 100,
    0
)

# Orden
resumen = resumen.sort_values(
    ["sku_madre", "ventas_canal"],
    ascending=[True, False]
).reset_index(drop=True)

print(resumen)

# =========================
# PIVOT OPCIONAL
# =========================
pivot_pct = (
    resumen.pivot_table(
        index="sku_madre",
        columns="canal",
        values="porcentaje_canal",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

pivot_ventas = (
    resumen.pivot_table(
        index="sku_madre",
        columns="canal",
        values="ventas_canal",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

# =========================
# EXPORTAR
# =========================
salida = "porcentaje_ventas_por_canal_ultimos_3_meses.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="detalle", index=False)
    pivot_pct.to_excel(writer, sheet_name="porcentajes", index=False)
    pivot_ventas.to_excel(writer, sheet_name="unidades", index=False)

print(f"Archivo generado: {salida}")

    sku_madre          canal  ventas_canal  ventas_totales_sku  \
0      IQ1005        ELEKTRA             1                   1   
1       IQ102         AMAZON             3                   3   
2      IQ1036         AMAZON             1                   1   
3      IQ1045         AMAZON             1                   1   
4      IQ1084         AMAZON             1                   1   
..        ...            ...           ...                 ...   
98      IQ889         AMAZON             1                   1   
99      IQ890         AMAZON             7                   7   
100     IQ897  MERCADO LIBRE             3                   3   
101     IQ902         AMAZON             5                   5   
102     IQ966      LIVERPOOL            18                  18   

     porcentaje_canal  
0               100.0  
1               100.0  
2               100.0  
3               100.0  
4               100.0  
..                ...  
98              100.0  
99             